Célula 1 - O Merge Espacial (União PRF + DNIT)

* O algoritmo da PRF anota o quilômetro exato do acidente (ex: 15.3), mas o DNIT avalia trechos inteiros. Para fazer essas bases conversarem, nós arredondamos a coluna de KM de ambas as tabelas (criando a km_join). 
* Em seguida, aplicamos um Left Join: o código pegou cada linha de acidente da PRF e "colou" ao lado dela as condições do asfalto que o DNIT registrou naquele mesmo trecho e rodovia.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Carregar as bases limpas
df_prf = pd.read_csv('../data/processed/prf_limpo.csv', sep=';', encoding='utf-8')
df_dnit = pd.read_csv('../data/processed/dnit_limpo.csv', sep=';', encoding='latin-1')

# 2. Padronização das Chaves de Cruzamento (BR e KM)
# A PRF geralmente usa a coluna 'br' (ex: 316). O DNIT usa 'rodovia' (ex: 'BR-316'). 
# Vamos extrair apenas os números do DNIT para bater com a PRF.
df_dnit['br'] = df_dnit['rodovia'].astype(str).str.extract(r'(\d+)').astype(float)
df_prf['br'] = df_prf['br'].astype(float)

# Arredondando os KMs para criar a chave de conexão espacial
df_prf['km_join'] = df_prf['km'].round()
df_dnit['km_join'] = df_dnit['km'].round()

# 3. O Merge (Left Join: Mantém todos os acidentes e traz a infraestrutura quando existir)
df_master = pd.merge(df_prf, df_dnit, on=['br', 'km_join'], how='left')

print(f"Base unificada criada! Total de registros: {len(df_master)}")
print(f"Colunas disponíveis: {df_master.shape[1]}")

Base unificada criada! Total de registros: 18052
Colunas disponíveis: 51


Célula 2 - Criação de Novas Features

* Modelos matemáticos não entendem o conceito de uma "Data" (ex: 25/12/2023). Nós quebramos essa data em pedaços que afetam a vida real: criamos uma coluna indicando o dia da semana, uma flag binária para saber se era fim de semana (1 para sim, 0 para não) e agrupamos as horas em turnos (Manhã, Tarde, Noite, Madrugada). 
* Isso permite que o modelo aprenda regras como: "acidentes de madrugada aos finais de semana tendem a ser mais graves".

In [6]:
# 1. Extração de Features Temporais
df_master['data_inversa'] = pd.to_datetime(df_master['data_inversa'], errors='coerce')

# Dia da semana (0 = Segunda, 6 = Domingo)
df_master['dia_semana'] = df_master['data_inversa'].dt.dayofweek

# Flag Binária: É fim de semana? (1 = Sim, 0 = Não)
df_master['is_fim_semana'] = df_master['dia_semana'].apply(lambda x: 1 if x >= 5 else 0)

# 2. Categorização de Horário (Turnos)
def categorizar_turno(hora):
    if pd.isna(hora): return 'Não Informado'
    if 6 <= hora < 12: return 'Manhã'
    elif 12 <= hora < 18: return 'Tarde'
    elif 18 <= hora < 24: return 'Noite'
    else: return 'Madrugada'

df_master['turno'] = df_master['hora_decimal'].apply(categorizar_turno)

print("Features temporais criadas com sucesso.")

Features temporais criadas com sucesso.


Célula 3 - Definição do Target e Features

1. O Target (Gabarito): Pegamos a coluna original que descrevia a gravidade e a transformamos em uma pergunta binária (0 = Acidente sem vítima; 1 = Acidente com vítima).
2. Busca Inteligente: Adicionamos uma linha de segurança para caçar a coluna superficie, independentemente de ela ter sido carregada com ou sem acento.
3. Filtro de Nulos: Agrupamos as features (variáveis) em texto e números, e passamos um filtro rigoroso (dropna). Bibliotecas de Machine Learning falham se virem um único espaço em branco. Essa etapa garantiu que só passassem registros 100% preenchidos.

In [7]:
# 1. Adaptando a coluna 'classificacao_acidente' para um problema binário
df_master['target_com_vitima'] = df_master['classificacao_acidente'].apply(
    lambda x: 0 if 'Sem Vítimas' in str(x) else 1
)

# 2. Busca dinâmica para lidar com problemas de acentuação na importação
coluna_superficie = next((col for col in df_master.columns if 'superf' in col), 'superficie')

# 3. Selecionando as features que vão alimentar o modelo
features_categoricas = ['condicao_metereologica', 'tipo_pista', 'turno', coluna_superficie, 'sentido']
features_numericas = ['hora_decimal', 'icm', 'ip', 'num_faixas']

# 4. Removendo registros nulos (necessário para o Scikit-Learn funcionar)
df_modelo = df_master.dropna(subset=features_categoricas + features_numericas + ['target_com_vitima']).copy()

X = df_modelo[features_categoricas + features_numericas]
y = df_modelo['target_com_vitima']

print(f"Features separadas com sucesso! Coluna de superfície identificada como: '{coluna_superficie}'")

Features separadas com sucesso! Coluna de superfície identificada como: 'superficie'


Célula 4 - Encoding e Normalização (Scikit-Learn)

* Usamos o `ColumnTransformer` para aplicar duas técnicas simultaneamente:
1. Codificação de variáveis categóricas: O algoritmo não sabe ler "Chuva" ou "Sol". O One-Hot Encoder transformou a coluna de clima em várias colunas de Zeros e Uns. Se choveu, a coluna clima_chuva recebe 1 e as outras recebem 0.  
2. Normalização/Padronização: A "hora" vai até 23, mas o índice do asfalto (ICM) geralmente é um número pequeno (ex: 1.5). Para o modelo não achar que a hora é mais importante só por ser um número maior, o StandardScaler espremeu todos os números para a mesma escala (com média 0 e desvio padrão 1), nivelando a importância de todas as grandezas.  
* O que Retornou: A matriz `df_X_final`.

In [8]:
# Definindo as transformações
transformador_categorico = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
transformador_numerico = StandardScaler()

# Criando o Pipeline de Processamento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, features_numericas),
        ('cat', transformador_categorico, features_categoricas)
    ])

# Aplicando as transformações em X
X_processado = preprocessor.fit_transform(X)

# Recuperando os nomes das colunas criadas pelo OneHotEncoder
nomes_cat = preprocessor.named_transformers_['cat'].get_feature_names_out(features_categoricas)
todas_features = features_numericas + list(nomes_cat)

# Transformando de volta em DataFrame para facilitar a visualização e modelagem
df_X_final = pd.DataFrame(X_processado, columns=todas_features)

print(f"Shape final dos dados processados: {df_X_final.shape}")
display(df_X_final.head(3))

Shape final dos dados processados: (14432, 18)


,hora_decimal,icm,ip,num_faixas,condicao_metereologica_chuva,condicao_metereologica_garoa/chuvisco,condicao_metereologica_ignorado,condicao_metereologica_nevoeiro/neblina,condicao_metereologica_nublado,condicao_metereologica_sol,tipo_pista_multipla,tipo_pista_simples,turno_Manhã,turno_Noite,turno_Tarde,superficie_4,superficie_Pavimentada,sentido_D
0,0.108125,1.273604,1.511168,1.722819,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.108125,1.273604,1.511168,1.722819,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.108125,1.273604,1.511168,1.722819,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
